# Phase 6C — self-contained validation audit

This notebook contains the audit implementation directly. Attach only the private Phase 5 validation bundle and private P1 checkpoint. It does not use the test set or train a model.

In [ ]:
import importlib.metadata as md, subprocess, sys
PINS={'transformers':'4.57.1','tokenizers':'0.22.2','huggingface-hub':'0.36.2','safetensors':'0.8.0'}
installed={}
for p in PINS:
    try: installed[p]=md.version(p)
    except md.PackageNotFoundError: installed[p]=None
print('Installed:', installed)
if installed != PINS:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--no-deps',*[f'{p}=={v}' for p,v in PINS.items()]],check=True)
    raise RuntimeError('Packages were pinned. Restart the Kaggle session and run all cells again.')
try:
    import tqdm
except ModuleNotFoundError:
    subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','tqdm'],check=True)
    raise RuntimeError('tqdm was installed. Restart the Kaggle session and run all cells again.')
print('tqdm:',tqdm.__version__)

In [ ]:
from pathlib import Path
import csv, copy, hashlib, json, re, time, unicodedata, zipfile
import soundfile as sf, torch
from tqdm.auto import tqdm
from transformers import GenerationConfig, WhisperForConditionalGeneration, WhisperProcessor
ROOT=Path('/kaggle/input'); OUT=Path('/kaggle/working/phase6c_private'); OUT.mkdir(parents=True,exist_ok=True)
def bundle_zips(kind):
    found=[]
    for p in ROOT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(p) as z:
                for n in z.namelist():
                    if n.endswith('bundle_record.json') and json.loads(z.read(n)).get('bundle_type')==kind: found.append(p)
        except (OSError,zipfile.BadZipFile,json.JSONDecodeError): pass
    return sorted(set(found))
def bundle_dirs(kind):
    found=[]
    for marker in ROOT.rglob('bundle_record.json'):
        try:
            if json.loads(marker.read_text()).get('bundle_type')==kind: found.append(marker.parent)
        except (OSError,json.JSONDecodeError): pass
    return sorted(set(found))
validation=bundle_zips('phase5_validation_baseline') or bundle_dirs('phase5_validation_baseline')
checkpoints=sorted(p for p in ROOT.rglob('*') if p.is_dir() and 'best_checkpoint' in p.name.lower() and any(p.rglob('model.safetensors')))
checkpoints += sorted(p for p in ROOT.rglob('*.zip') if 'best_checkpoint' in p.name.lower())
if len(validation)!=1 or len(checkpoints)!=1:
    print('Mounted input paths:'); [print(' -',p) for p in sorted(ROOT.iterdir())]
    raise FileNotFoundError('Attach exactly one private validation bundle and one best-checkpoint dataset or ZIP.')
validation, checkpoint = validation[0], checkpoints[0]
print({'validation':validation,'checkpoint':checkpoint,'output':OUT})
assert 'test' not in validation.name.lower() and 'test' not in checkpoint.name.lower()

In [ ]:
MODEL_ID='neurlang/ipa-whisper-base'; REV='949f7c4e4ee374b5202bc5886f6e4552e5ae2371'; EOS=50257; MAXLEN=448
work=OUT/'_extracted'; work.mkdir(exist_ok=True)
def extract(archive,dest):
    if archive.is_dir():
        return archive, json.loads((archive/'bundle_record.json').read_text())
    with zipfile.ZipFile(archive) as z:
        names=z.namelist(); marker=next(n for n in names if n.endswith('bundle_record.json')); rec=json.loads(z.read(marker)); root=dest/rec['bundle_type']; root.mkdir(parents=True,exist_ok=True)
        for m in tqdm(names,desc='Extracting validation bundle',unit='file'): z.extract(m,root)
    return root,rec
vroot,vrec=extract(validation,work/'validation')
assert vrec['evaluation_partition']=='validation' and vrec['clips']==732 and not vrec['contains_test_audio']
if checkpoint.is_dir():
    croot=checkpoint
else:
    with zipfile.ZipFile(checkpoint) as z:
        croot=work/'checkpoint'; croot.mkdir(exist_ok=True); [z.extract(m,croot) for m in tqdm(z.namelist(),desc='Extracting checkpoint',unit='file')]
models=[p.parent for p in croot.rglob('config.json') if (p.parent/'model.safetensors').is_file()]; assert len(models)==1
model_dir=models[0]
rows=list(csv.DictReader((vroot/'validation_manifest.csv').open(newline='',encoding='utf-8'))); assert len(rows)==732
processor=WhisperProcessor.from_pretrained(MODEL_ID,revision=REV)
model=WhisperForConditionalGeneration.from_pretrained(model_dir,torch_dtype=torch.float32,attn_implementation='sdpa').cuda().eval(); model.config.use_cache=True
g=GenerationConfig.from_dict(json.loads((vroot/'generation_config.json').read_text())); g.forced_decoder_ids=[[1,None],[2,50359]]; g.language=None; g.task=None; g.suppress_tokens=[]; g.begin_suppress_tokens=[220,EOS]; g.do_sample=False; g.num_beams=1; g.max_length=MAXLEN; g.return_dict_in_generate=True
def norm(s): return re.sub(r' {2,}',' ',unicodedata.normalize('NFD',str(s)).strip())
def edits(a,b):
    x,y=list(a),list(b); d=[[0]*(len(y)+1) for _ in range(len(x)+1)]
    for i in range(len(x)+1): d[i][0]=i
    for j in range(len(y)+1): d[0][j]=j
    for i in range(1,len(x)+1):
        for j in range(1,len(y)+1): d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,d[i-1][j-1]+(x[i-1]!=y[j-1]))
    s=de=ins=0; i,j=len(x),len(y)
    while i or j:
        if i and j and x[i-1]==y[j-1] and d[i][j]==d[i-1][j-1]: i-=1;j-=1
        elif i and j and d[i][j]==d[i-1][j-1]+1: s+=1;i-=1;j-=1
        elif i and d[i][j]==d[i-1][j]+1: de+=1;i-=1
        else: ins+=1;j-=1
    return s,de,ins
ordered=sorted(enumerate(rows),key=lambda z:(float(z[1]['duration_seconds']),z[1]['sample_id'])); chosen=[ordered[round(i*(len(ordered)-1)/11)][0] for i in range(12)]; selected=[rows[i] for i in chosen]
def feat(r):
    a,sr=sf.read((vroot/r['audio_file']).resolve(),dtype='float32',always_2d=True); assert sr==16000 and a.shape[1]==1; return processor.feature_extractor(a[:,0],sampling_rate=16000,return_tensors='pt').input_features[0]
features=[feat(r) for r in tqdm(selected,desc='Extracting audit features',unit='clip')]
def generate(fs,rs):
    with torch.inference_mode(): out=model.generate(input_features=torch.stack(fs).cuda(),generation_config=copy.deepcopy(g))
    result=[]
    for r,seq in zip(rs,out.sequences.detach().cpu().tolist()):
        ids=seq[:seq.index(EOS)+1] if EOS in seq else seq[:MAXLEN]; raw=processor.tokenizer.decode(ids,skip_special_tokens=True); hyp=norm(raw); ref=norm(r['reference']); s,de,ins=edits(ref,hyp); result.append({'sample_id':r['sample_id'],'reference':ref,'raw_prediction':raw,'normalized_prediction':hyp,'token_ids':ids,'output_length':len(ids),'eos':bool(ids and ids[-1]==EOS),'cap_without_eos':len(ids)>=MAXLEN and (not ids or ids[-1]!=EOS),'S':s,'D':de,'I':ins,'duration_seconds':float(r['duration_seconds'])})
    return result
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t=time.perf_counter(); single=[]
for i in tqdm(range(12),desc='Single-clip generation',unit='clip'): single += generate([features[i]],[selected[i]])
torch.cuda.synchronize(); single_seconds=time.perf_counter()-t; single_mem=torch.cuda.max_memory_allocated()/1024**2
comparisons=[]
for bs in tqdm([4,8],desc='Testing batch sizes',unit='batch size'):
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); t=time.perf_counter(); batched=[]
    try:
        for i in tqdm(range(0,12,bs),desc=f'Batch-{bs} generation',leave=False,unit='batch'): batched += generate(features[i:i+bs],selected[i:i+bs])
        same=all(a['token_ids']==b['token_ids'] and a['raw_prediction']==b['raw_prediction'] and a['normalized_prediction']==b['normalized_prediction'] for a,b in zip(single,batched)) and [(sum(x[k] for x in single)) for k in ('S','D','I')]==[(sum(x[k] for x in batched)) for k in ('S','D','I')]
        status='pass' if same else 'mismatch'; err=None
    except torch.cuda.OutOfMemoryError as e: torch.cuda.empty_cache(); status='oom'; err=str(e)
    torch.cuda.synchronize(); sec=time.perf_counter()-t; mem=torch.cuda.max_memory_allocated()/1024**2; comparisons.append({'batch_size':bs,'status':status,'runtime_seconds':sec,'peak_allocated_mib':mem,'speedup_vs_single':single_seconds/sec,'error':err}); print(comparisons[-1])
passed=[x for x in comparisons if x['status']=='pass']; assert passed, comparisons; best=max(passed,key=lambda x:x['batch_size'])
def agg(rs): return {'clips':len(rs),'reference_codepoints':sum(len(x['reference']) for x in rs),'S':sum(x['S'] for x in rs),'D':sum(x['D'] for x in rs),'I':sum(x['I'] for x in rs),'eos':sum(x['eos'] for x in rs),'cap_without_eos':sum(x['cap_without_eos'] for x in rs)}
out={'status':'PASS','single':{'runtime_seconds':single_seconds,'peak_allocated_mib':single_mem,'aggregate':agg(single)},'batched':comparisons,'chosen_batch_size':best['batch_size'],'generation_policy':g.to_dict(),'test_data_used':False}
(OUT/'phase6c_equivalence.json').write_text(json.dumps(out,indent=2)+'\n'); (OUT/'phase6c_diagnostics_private.jsonl').write_text('\n'.join(json.dumps(x,ensure_ascii=False) for x in single)+'\n'); print(json.dumps(out,indent=2)); print('PHASE 6C PASS; chosen batch:',best['batch_size'])